# 1. Data Preparation
Loading and merging CERF, CBPF, and INFORM Severity data.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# Constants
COLOR_BLUE = '#1f77b4'
COLOR_GREEN = 'green'

# 1. Load and Aggregate CERF Funding
cerf = pd.read_csv('Data_ CERF Donor Contributions and Allocations - allocations.csv')
cerf_agg = cerf.groupby(['countryCode', 'year'])['totalAmountApproved'].sum().reset_index()
cerf_agg.columns = ['ISO3', 'Year', 'CERF_Funding']

# 2. Load and Aggregate CBPF Funding
cbpf = pd.read_csv('Data_ Country Based Pooled Funds (CBPF) - Projects.csv')
cbpf['ISO3'] = cbpf['ChfProjectCode'].str.split('-').str[0]
cbpf_agg = cbpf.groupby(['ISO3', 'AllocationYear'])['Budget'].sum().reset_index()
cbpf_agg.columns = ['ISO3', 'Year', 'CBPF_Funding']

# 3. Load and Clean INFORM Severity Data
inform = pd.read_csv('inform_severity_cleaned.csv')
inform = inform[['ISO3', 'Year', 'INFORM Severity Index']].dropna()
inform_agg = inform.groupby(['ISO3', 'Year'])['INFORM Severity Index'].max().reset_index()

# 4. Merge All Data
df = inform_agg.merge(cerf_agg, on=['ISO3', 'Year'], how='left')
df = df.merge(cbpf_agg, on=['ISO3', 'Year'], how='left')
df[['CERF_Funding', 'CBPF_Funding']] = df[['CERF_Funding', 'CBPF_Funding']].fillna(0)
df['Total_Actual_Funding'] = df['CERF_Funding'] + df['CBPF_Funding']

# Filter for relevant years (2020-2025)
df_clean = df[df['Year'] >= 2020].copy()
print(f"Dataset ready with {len(df_clean)} records.")

# 2. Exploratory Data Analysis (EDA)
Understanding the distribution of our key variables: Severity and Funding.

In [ ]:
# Distribution Plots
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# 1. Severity Distribution
sns.histplot(df_clean['INFORM Severity Index'], bins=20, kde=True, ax=axes[0], color=COLOR_BLUE)
axes[0].set_title('Distribution of INFORM Severity Index')
axes[0].set_xlabel('Severity Index (0-5)')

# 2. Funding Distribution (Log Scale)
# Use log scale because funding is highly right-skewed
sns.histplot(df_clean['Total_Actual_Funding'], bins=30, ax=axes[1], color='green')
axes[1].set_xscale('log')
axes[1].set_title('Distribution of Funding (Log Scale)')
axes[1].set_xlabel('Total Actual Funding (USD)')

plt.tight_layout()
plt.show()

# 3. Correlation Analysis
Examining the relationship between severity and funding.

In [ ]:
plt.figure(figsize=(12, 7))
sns.scatterplot(data=df_clean, x='INFORM Severity Index', y='Total_Actual_Funding', hue='Year', palette='viridis', alpha=0.6)
plt.yscale('log')
plt.title('Crisis Severity vs Total Pooled Funding (Log Scale)', fontsize=14)
plt.xlabel('INFORM Severity Index', fontsize=12)
plt.ylabel('Total Actual Funding (USD, Log Scale)', fontsize=12)
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.legend(title='Year', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

# 4. Model Training
Building a log-linear regression model to predict funding based on severity.

In [ ]:
# Prepare for Modeling (Log-Linear)
df_model = df_clean[df_clean['Total_Actual_Funding'] > 0].copy()
X = df_model[['INFORM Severity Index']]
y = np.log1p(df_model['Total_Actual_Funding'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred_log = model.predict(X_test)
r2 = r2_score(y_test, y_pred_log)

print(f"Intercept (Log): {model.intercept_:.2f}")
print(f"Coefficient: {model.coef_[0]:.2f}")
print(f"R-squared (Test): {r2:.4f}")
print(f"R-squared (Full): {model.score(X, y):.4f}")

# 5. Funding Gap Analysis
Identifying under-funded crises based on model predictions.

In [ ]:
# Calculate Residuals and Gaps (Back-transformed)
df_model['Predicted_Funding_Log'] = model.predict(df_model[['INFORM Severity Index']])
df_model['Predicted_Funding'] = np.expm1(df_model['Predicted_Funding_Log'])
df_model['Residual'] = df_model['Total_Actual_Funding'] - df_model['Predicted_Funding']
df_model['Funding_Gap_Million'] = df_model['Residual'] / 1e6

# Labeling based on negative residuals (Under-funded)
df_model['Status'] = np.where(df_model['Residual'] < 0, 'Under-funded', 'Above-average')

print("Top 5 Under-funded Crises by Dollar Gap:")
display(df_model[df_model['Status'] == 'Under-funded'].sort_values('Funding_Gap_Million').head())

# 6. Visualization: Actual vs Predicted Funding
Comparing actual funding against model predictions.

In [ ]:
plt.figure(figsize=(12, 7))
plt.scatter(df_model['INFORM Severity Index'], df_model['Total_Actual_Funding'], 
            alpha=0.5, label='Actual Funding', color='blue')
plt.scatter(df_model['INFORM Severity Index'], df_model['Predicted_Funding'], 
            alpha=0.5, label='Predicted Funding', color='red', marker='x')
plt.yscale('log')
plt.xlabel('INFORM Severity Index', fontsize=12)
plt.ylabel('Funding (USD, Log Scale)', fontsize=12)
plt.title('Actual vs Predicted Funding by Severity', fontsize=14)
plt.legend()
plt.grid(True, which="both", ls="--", alpha=0.3)
plt.tight_layout()
plt.show()